# Substitution Counterfactual Analysis

> These are Go notebooks: In order to use the GoNB Jupyter Kernel, please install GoNB from here: https://github.com/janpfeifer/gonb

Note also that for local package development, you can put: `!*go mod edit -replace "github.com/umbralcalc/trywizard=/path/to/trywizard"` at the top of any cell.

This notebook trains a substitution covariate model across all games, then simulates counterfactual substitution strategies to estimate their effect on win probability.

**Pipeline:**
1. Train multi-game baseline + covariate Poisson GLM coefficients
2. Define substitution strategies (timing per position group)
3. Monte Carlo simulate each strategy
4. Compare win probabilities across strategies

In [ ]:
import (
	"fmt"
	"math"

	"github.com/umbralcalc/trywizard/pkg/match"
	"github.com/go-echarts/go-echarts/v2/charts"
	"github.com/go-echarts/go-echarts/v2/opts"
	gonb_echarts "github.com/janpfeifer/gonb-echarts"
)

%%

// === Phase 1: Multi-game coefficient training ===

eventsPath := "/Users/roberth/Code/trywizard/dat/events.csv"
playersPath := "/Users/roberth/Code/trywizard/dat/players.csv"

fmt.Println("Training multi-game baseline + covariate model...")
coeffs, err := match.RunMultiGameBaselineCovariateTraining(
	eventsPath, playersPath,
	0.1,  // learning rate
	10,   // descent iterations
	50,   // window depth
)
if err != nil {
	panic(err)
}

_, err = match.ComputeSmoothedBaselineRates(eventsPath)
if err != nil {
	panic(err)
}

// Use average conversion probabilities across all games.
games, _ := match.ListGames(playersPath)
var totalHomeConvP, totalAwayConvP float64
nGames := 0
for _, g := range games {
	s, err := match.TransformEventsToStateTimeStorage(eventsPath, g.GameID, g.HomeTeamID)
	if err != nil {
		continue
	}
	cp := match.ComputeConversionProbabilities(s)
	totalHomeConvP += cp[0]
	totalAwayConvP += cp[1]
	nGames++
}
convProbs := []float64{totalHomeConvP / float64(nGames), totalAwayConvP / float64(nGames)}
fmt.Printf("Average conversion probabilities: home=%.3f, away=%.3f\n", convProbs[0], convProbs[1])

_, _ = match.SplitCoefficients(coeffs)

// --- Print coefficient table ---

rateLabels := []string{"home_try", "away_try", "home_penalty", "away_penalty", "home_yellow", "away_yellow"}
covLabels := []string{"intercept", "home_front", "home_back", "home_halves", "home_outside",
	"away_front", "away_back", "away_halves", "away_outside"}

fmt.Println("\nFitted coefficients (multi-game baseline + covariates):")
for i := 0; i < match.RateEventWidth; i++ {
	fmt.Printf("\n  %s:\n", rateLabels[i])
	for j := 0; j < match.CoeffsPerRate; j++ {
		fmt.Printf("    %-14s %+.6f\n", covLabels[j], coeffs[i*match.CoeffsPerRate+j])
	}
}

// --- Chart: Coefficient heatmap as grouped bar ---

bar := charts.NewBar()
bar.SetGlobalOptions(
	charts.WithTitleOpts(opts.Title{Title: "Substitution Covariate Coefficients by Rate Type"}),
	charts.WithTooltipOpts(opts.Tooltip{Trigger: "axis"}),
	charts.WithLegendOpts(opts.Legend{Show: opts.Bool(true)}),
	charts.WithYAxisOpts(opts.YAxis{Name: "Coefficient"}),
)
bar.SetXAxis(rateLabels)

// One series per covariate (skip intercept, show the 8 sub covariates).
for j := 1; j < match.CoeffsPerRate; j++ {
	vals := make([]opts.BarData, match.RateEventWidth)
	for i := 0; i < match.RateEventWidth; i++ {
		vals[i] = opts.BarData{Value: math.Round(coeffs[i*match.CoeffsPerRate+j]*10000) / 10000}
	}
	bar.AddSeries(covLabels[j], vals)
}
gonb_echarts.Display(bar, "width: 1024px; height: 500px; background: white;")

In [ ]:
import (
	"fmt"
	"math"

	"github.com/umbralcalc/trywizard/pkg/match"
	"github.com/go-echarts/go-echarts/v2/charts"
	"github.com/go-echarts/go-echarts/v2/opts"
	gonb_echarts "github.com/janpfeifer/gonb-echarts"
)

%%

// === Phase 2: Strategy comparison ===

eventsPath := "/Users/roberth/Code/trywizard/dat/events.csv"
playersPath := "/Users/roberth/Code/trywizard/dat/players.csv"

// Re-train (each cell is independent in GoNB).
coeffs, err := match.RunMultiGameBaselineCovariateTraining(
	eventsPath, playersPath, 0.1, 10, 50,
)
if err != nil {
	panic(err)
}
baselineRates, _ := match.ComputeSmoothedBaselineRates(eventsPath)
scoreCoeffs, cardCoeffs := match.SplitCoefficients(coeffs)

games, _ := match.ListGames(playersPath)
var totalHomeConvP, totalAwayConvP float64
nGames := 0
for _, g := range games {
	s, err := match.TransformEventsToStateTimeStorage(eventsPath, g.GameID, g.HomeTeamID)
	if err != nil {
		continue
	}
	cp := match.ComputeConversionProbabilities(s)
	totalHomeConvP += cp[0]
	totalAwayConvP += cp[1]
	nGames++
}
convProbs := []float64{totalHomeConvP / float64(nGames), totalAwayConvP / float64(nGames)}

const nSims = 500
const nSteps = 80

// Define strategies to compare.
// The away team uses a fixed "typical" strategy throughout.
awaySubs := [match.NumPositionGroups]int{50, 55, 60, 65}

type namedStrategy struct {
	Name     string
	Strategy *match.SubstitutionStrategy
}

strategies := []namedStrategy{
	{"No subs", &match.SubstitutionStrategy{
		HomeSubs: [match.NumPositionGroups]int{0, 0, 0, 0},
		AwaySubs: awaySubs,
	}},
	{"Early (40-55)", &match.SubstitutionStrategy{
		HomeSubs: [match.NumPositionGroups]int{40, 45, 50, 55},
		AwaySubs: awaySubs,
	}},
	{"Standard (50-65)", &match.SubstitutionStrategy{
		HomeSubs: [match.NumPositionGroups]int{50, 55, 60, 65},
		AwaySubs: awaySubs,
	}},
	{"Late (60-75)", &match.SubstitutionStrategy{
		HomeSubs: [match.NumPositionGroups]int{60, 65, 70, 75},
		AwaySubs: awaySubs,
	}},
	{"Front row only", &match.SubstitutionStrategy{
		HomeSubs: [match.NumPositionGroups]int{50, 0, 0, 0},
		AwaySubs: awaySubs,
	}},
	{"Backs first", &match.SubstitutionStrategy{
		HomeSubs: [match.NumPositionGroups]int{65, 60, 45, 40},
		AwaySubs: awaySubs,
	}},
}

// Run simulations for each strategy.
type strategyResult struct {
	Name  string
	Probs match.WinProbabilities
	MeanHomeFinal float64
	MeanAwayFinal float64
}

results := make([]strategyResult, len(strategies))
for i, ns := range strategies {
	fmt.Printf("Simulating '%s' (%d subs)...\n", ns.Name, ns.Strategy.TotalSubs())
	simResults := match.RunStrategySimulations(
		scoreCoeffs, cardCoeffs, convProbs,
		baselineRates, ns.Strategy,
		nSims, nSteps, uint64(i*10000),
	)
	probs := match.ComputeWinProbabilities(simResults)

	var sumH, sumA float64
	for _, r := range simResults {
		if len(r.ScoreTrajectory) > 0 {
			last := r.ScoreTrajectory[len(r.ScoreTrajectory)-1]
			sumH += last.Home
			sumA += last.Away
		}
	}

	results[i] = strategyResult{
		Name:  ns.Name,
		Probs: probs,
		MeanHomeFinal: sumH / float64(nSims),
		MeanAwayFinal: sumA / float64(nSims),
	}
}

// --- Print results table ---

fmt.Printf("\n%-20s %8s %8s %8s %10s %10s\n",
	"Strategy", "P(Home)", "P(Draw)", "P(Away)", "Mean Home", "Mean Away")
fmt.Println("------------------------------------------------------------------------")
for _, r := range results {
	fmt.Printf("%-20s %8.3f %8.3f %8.3f %10.1f %10.1f\n",
		r.Name, r.Probs.HomeWin, r.Probs.Draw, r.Probs.AwayWin,
		r.MeanHomeFinal, r.MeanAwayFinal)
}

// --- Chart: Win probabilities by strategy ---

stratNames := make([]string, len(results))
for i, r := range results {
	stratNames[i] = r.Name
}

bar := charts.NewBar()
bar.SetGlobalOptions(
	charts.WithTitleOpts(opts.Title{Title: "Home Win Probability by Substitution Strategy"}),
	charts.WithTooltipOpts(opts.Tooltip{Trigger: "axis"}),
	charts.WithLegendOpts(opts.Legend{Show: opts.Bool(true)}),
	charts.WithYAxisOpts(opts.YAxis{Name: "Probability", Max: "1.0"}),
)
bar.SetXAxis(stratNames)

homeWinBars := make([]opts.BarData, len(results))
drawBars := make([]opts.BarData, len(results))
awayWinBars := make([]opts.BarData, len(results))
for i, r := range results {
	homeWinBars[i] = opts.BarData{Value: math.Round(r.Probs.HomeWin*1000) / 1000}
	drawBars[i] = opts.BarData{Value: math.Round(r.Probs.Draw*1000) / 1000}
	awayWinBars[i] = opts.BarData{Value: math.Round(r.Probs.AwayWin*1000) / 1000}
}
bar.AddSeries("Home Win", homeWinBars)
bar.AddSeries("Draw", drawBars)
bar.AddSeries("Away Win", awayWinBars)
gonb_echarts.Display(bar, "width: 1024px; height: 500px; background: white;")

In [ ]:
import (
	"fmt"

	"github.com/umbralcalc/trywizard/pkg/match"
	"github.com/go-echarts/go-echarts/v2/charts"
	"github.com/go-echarts/go-echarts/v2/opts"
	gonb_echarts "github.com/janpfeifer/gonb-echarts"
)

%%

// === Phase 2b: Sweep substitution timing for one position group ===
// Hold all other home subs fixed at minute 55, sweep outside backs from 40-75.

eventsPath := "/Users/roberth/Code/trywizard/dat/events.csv"
playersPath := "/Users/roberth/Code/trywizard/dat/players.csv"

coeffs, err := match.RunMultiGameBaselineCovariateTraining(
	eventsPath, playersPath, 0.1, 10, 50,
)
if err != nil {
	panic(err)
}
baselineRates, _ := match.ComputeSmoothedBaselineRates(eventsPath)
scoreCoeffs, cardCoeffs := match.SplitCoefficients(coeffs)

games, _ := match.ListGames(playersPath)
var totalHomeConvP, totalAwayConvP float64
nGames := 0
for _, g := range games {
	s, err := match.TransformEventsToStateTimeStorage(eventsPath, g.GameID, g.HomeTeamID)
	if err != nil {
		continue
	}
	cp := match.ComputeConversionProbabilities(s)
	totalHomeConvP += cp[0]
	totalAwayConvP += cp[1]
	nGames++
}
convProbs := []float64{totalHomeConvP / float64(nGames), totalAwayConvP / float64(nGames)}

const nSims = 500
const nSteps = 80
awaySubs := [match.NumPositionGroups]int{50, 55, 60, 65}

// Sweep each position group independently.
groupNames := []string{"Front Row", "Back Row", "Halves", "Outside Backs"}
sweepMinutes := []int{0, 40, 45, 50, 55, 60, 65, 70, 75}
baseHomeSubs := [match.NumPositionGroups]int{50, 55, 55, 55}

// sweepResults[group][minuteIdx] = homeWinProb
sweepResults := make([][]float64, match.NumPositionGroups)

for g := 0; g < match.NumPositionGroups; g++ {
	sweepResults[g] = make([]float64, len(sweepMinutes))
	for mi, minute := range sweepMinutes {
		homeSubs := baseHomeSubs
		homeSubs[g] = minute
		strategy := &match.SubstitutionStrategy{
			HomeSubs: homeSubs,
			AwaySubs: awaySubs,
		}
		simResults := match.RunStrategySimulations(
			scoreCoeffs, cardCoeffs, convProbs,
			baselineRates, strategy,
			nSims, nSteps, uint64(g*1000+mi*100),
		)
		probs := match.ComputeWinProbabilities(simResults)
		sweepResults[g][mi] = probs.HomeWin
		fmt.Printf("  %s @ %d min: P(home win)=%.3f\n", groupNames[g], minute, probs.HomeWin)
	}
}

// --- Chart: Win probability vs substitution minute per group ---

xLabels := make([]string, len(sweepMinutes))
for i, m := range sweepMinutes {
	if m == 0 {
		xLabels[i] = "Never"
	} else {
		xLabels[i] = fmt.Sprintf("%d", m)
	}
}

toLD := func(v []float64) []opts.LineData {
	d := make([]opts.LineData, len(v))
	for i, x := range v {
		d[i] = opts.LineData{Value: x}
	}
	return d
}

line := charts.NewLine()
line.SetGlobalOptions(
	charts.WithTitleOpts(opts.Title{Title: "Home Win Probability vs Substitution Minute"}),
	charts.WithXAxisOpts(opts.XAxis{Name: "Substitution Minute"}),
	charts.WithYAxisOpts(opts.YAxis{Name: "P(Home Win)"}),
	charts.WithTooltipOpts(opts.Tooltip{Trigger: "axis"}),
	charts.WithLegendOpts(opts.Legend{Show: opts.Bool(true)}),
)
line.SetXAxis(xLabels)
for g := 0; g < match.NumPositionGroups; g++ {
	line.AddSeries(groupNames[g], toLD(sweepResults[g]),
		charts.WithLineStyleOpts(opts.LineStyle{Width: 2}))
}
gonb_echarts.Display(line, "width: 1024px; height: 500px; background: white;")